# IOE421 · Large Language Models
# Training Language Models to Reason Efficiently — Qwen2.5-3B Edition
**Implementing arXiv:2502.04463 | New Model Variant**

**Team:** Sathvik Kiran (2022BCD0028) · M Naresh (2022BCD0024) · Srivathsa K (2022BCD0020)

---

## Model: `Qwen/Qwen2.5-3B-Instruct`
- 3B parameters — fits on T4 GPU with headroom for RL training
- Strong math reasoning baseline (outperforms Mistral-7B on GSM8K zero-shot)
- Native chat template compatible with `apply_chat_template`
- Efficient architecture: GQA + RoPE + SwiGLU

**Pipeline:**
1. Stage 1 — SFT with Unsloth LoRA
2. Stage 2 — PPO + RLOO + Length-Penalty RL
3. Stage 3 — DAC + MORS Extensions
4. Stage 4 — Full Evaluation (5 metrics)
5. Stage 5 — Visualisations & Inference Demo

## Before You Start
1. **GPU Required**: `Runtime` > `Change runtime type` > **T4 GPU**
2. **Run All**: `Runtime` > `Run all`


In [1]:
%%capture
!pip install unsloth
!pip install 'datasets>=2.14' trl transformers accelerate bitsandbytes peft
!pip install sentence-transformers matplotlib nltk
print("All packages installed")


In [2]:
import torch
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'   Memory: {gpu.total_memory / 1024**3:.1f} GB')
    print(f'   Compute capability: {gpu.major}.{gpu.minor}')
else:
    print('WARNING: No GPU! Go to Runtime > Change runtime type > T4 GPU')


GPU: Tesla T4
   Memory: 14.6 GB
   Compute capability: 7.5


## HuggingFace Login (Optional)


In [3]:
# from huggingface_hub import login
# login()  # Token at: https://huggingface.co/settings/tokens


---
## Stage 1 · Load Dataset

300 GSM8K math conversations (same TuneKit dataset as reference notebook).


In [4]:
import base64, json

with open("dataset.txt", "r") as f:
    DATASET_B64 = f.read()

raw = base64.b64decode(DATASET_B64.strip()).decode('utf-8')
conversations = [json.loads(line) for line in raw.strip().split('\n') if line.strip()]
print(f'Loaded {len(conversations)} conversations')
print(f'Sample Q: {conversations[0]["messages"][1]["content"][:80]}...')
print(f'Sample A: {conversations[0]["messages"][2]["content"][:80]}...')

Loaded 300 conversations
Sample Q: Natalia sold clips to 48 of her friends in April, and then she sold half as many...
Sample A: Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>...


---
## Stage 2A · SFT — Qwen2.5-3B-Instruct + Unsloth LoRA

**Why Qwen2.5-3B?**
- 3B params fit comfortably on T4 (14.6 GB) even with RL training overhead
- Native strong math reasoning — ideal for efficient reasoning training
- LoRA targets: `q_proj k_proj v_proj o_proj gate_proj up_proj down_proj`


In [5]:
import os, re, math, time, random
import numpy as np
from collections import defaultdict
from typing import List, Dict, Tuple, Optional

os.environ['TORCHDYNAMO_DISABLE'] = '1'

from unsloth import FastLanguageModel
import torch
import torch.nn.functional as F

torch.manual_seed(42)
random.seed(42)
np.random.seed(42)

MODEL_ID       = 'Qwen/Qwen2.5-3B-Instruct'
MAX_SEQ_LENGTH = 2048
DEVICE         = 'cuda' if torch.cuda.is_available() else 'cpu'

SYSTEM_PROMPT = (
    'You are a mathematical reasoning assistant. Always solve problems step by step. '
    'Keep reasoning concise. End your response with the final numeric answer. '
    'After the final answer, output <STOP>.'
)

print(f'Loading {MODEL_ID} with Unsloth (4-bit)...')

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

# Ensure pad token is set and embedding is resized BEFORE adding LoRA
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '<|pad_token|>'})
model.resize_token_embeddings(len(tokenizer))

model = FastLanguageModel.get_peft_model(
    model,
    r=16, lora_alpha=16, lora_dropout=0,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Qwen2.5-3B loaded! Vocab size: {len(tokenizer)}')
print(f'Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading Qwen/Qwen2.5-3B-Instruct with Unsloth (4-bit)...
==((====))==  Unsloth 2026.3.18: Fast Qwen2 patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.36G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.3.18 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


Qwen2.5-3B loaded! Vocab size: 151666
Trainable: 29,933,568 / 1,829,502,976 (1.64%)


In [6]:
from datasets import Dataset

def format_conversation(example):
    messages = example.get('messages', [])
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    return {'text': text}

dataset = Dataset.from_list(conversations)
dataset = dataset.map(format_conversation, remove_columns=dataset.column_names)
print(f'Dataset prepared: {len(dataset)} examples')
print(f'Sample (first 300 chars):\n{dataset[0]["text"][:300]}...')


Map:   0%|          | 0/300 [00:00<?, ? examples/s]

Dataset prepared: 300 examples
Sample (first 300 chars):
<|im_start|>system
You are a mathematical reasoning assistant. Always solve problems step by step. Keep reasoning concise. End your response with the final numeric answer. After the final answer, output <STOP>.<|im_end|>
<|im_start|>user
Natalia sold clips to 48 of her friends in April, and then she...


In [7]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

# Reduced max_seq_length to 1024 to save memory
MAX_SEQ_LENGTH_SFT = 1024

training_args = TrainingArguments(
    output_dir='./results_qwen',
    num_train_epochs=1,
    per_device_train_batch_size=1,      # Reduced from 2 to 1
    gradient_accumulation_steps=8,     # Increased from 4 to 8 to keep batch size 8
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_steps=5,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    logging_steps=10,
    save_strategy='epoch',
    save_total_limit=2,
    optim='adamw_8bit',
    lr_scheduler_type='linear',
    seed=42,
    report_to='none',
    gradient_checkpointing=True,       # Explicitly enabled for memory saving
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LENGTH_SFT,
    packing=False,
    args=training_args,
)

gpu_stats = torch.cuda.get_device_properties(0)
start_mem = round(torch.cuda.max_memory_reserved() / 1024**3, 2)
max_mem   = round(gpu_stats.total_memory / 1024**3, 2)
print(f'GPU: {gpu_stats.name}  |  Memory: {start_mem} GB / {max_mem} GB')
print('Starting Qwen2.5-3B SFT training with memory optimizations...')
print('=' * 50)

# Clear cache before starting
torch.cuda.empty_cache()

sft_stats = trainer.train()

end_mem = round(torch.cuda.max_memory_reserved() / 1024**3, 2)
print('=' * 50)
print(f'Training complete!')
print(f'   Peak GPU: {end_mem} GB ({round(end_mem/max_mem*100,1)}%)')
print(f'   Final loss: {sft_stats.training_loss:.4f}')

SFT_OUTPUT = './fine_tuned_qwen25_3b'
model.save_pretrained(SFT_OUTPUT)
tokenizer.save_pretrained(SFT_OUTPUT)
print(f'Checkpoint saved -> {SFT_OUTPUT}')

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/300 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


GPU: Tesla T4  |  Memory: 2.96 GB / 14.56 GB
Starting Qwen2.5-3B SFT training with memory optimizations...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 300 | Num Epochs = 3 | Total steps = 114
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 29,933,568 of 3,115,319,296 (0.96% trained)


Step,Training Loss
10,1.459185
20,0.378049
30,0.299346
40,0.255129
50,0.219647
60,0.214569
70,0.204491
80,0.198528
90,0.153140
100,0.146459


/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(


Training complete!
   Peak GPU: 5.22 GB (35.9%)
   Final loss: 0.3285


/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(


Checkpoint saved -> ./fine_tuned_qwen25_3b


In [8]:
FastLanguageModel.for_inference(model)

def qwen_generate(prompt, max_new_tokens=256):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': prompt},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(DEVICE)

    # Safety check for token IDs
    if (inputs['input_ids'] >= model.config.vocab_size).any():
        return "Error: Input contains Out-of-Vocabulary tokens."

    with torch.no_grad():
        outputs = model.generate(
          input_ids=inputs['input_ids'],
          attention_mask=inputs['attention_mask'],
          max_new_tokens=max_new_tokens,
          do_sample=True,
          pad_token_id=tokenizer.pad_token_id,
          eos_token_id=tokenizer.eos_token_id,
          use_cache=False,
      )
    return tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

print('=== Qwen2.5-3B SFT Inference Test ===')
try:
    for p in [
        'Natalia sold clips to 48 of her friends in April, then half as many in May. Total?',
        'Weng earns $12/hr babysitting. She did 50 minutes. How much did she earn?',
    ]:
        print(f'\nQ: {p}')
        print(f'A: {qwen_generate(p)}')
        print('-' * 60)
except Exception as e:
    print(f"Caught error: {e}")

Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== Qwen2.5-3B SFT Inference Test ===

Q: Natalia sold clips to 48 of her friends in April, then half as many in May. Total?


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API i

A: In April, Natalia sold 48 clips.
In May, she sold half as many, which is 48/2 = <<48/2=24>>24 clips.
The total number of clips Natalia sold is 48+24 = <<48+24=72>>72 clips.
#### 72 <STOP>
------------------------------------------------------------

Q: Weng earns $12/hr babysitting. She did 50 minutes. How much did she earn?
A: She earned $12 * .83 = $<<12*.83=9.96>>9.96
#### 9.96 <STOP>
------------------------------------------------------------


In [9]:
#@title Export to GGUF — Optional
run_gguf = False  #@param {type:"boolean"}

if run_gguf:
    model.save_pretrained_gguf('./qwen25_3b_gguf', tokenizer, quantization_method='q4_k_m')
    print('GGUF export complete')


---
## Stage 2B · Reinforcement Learning Loop
### PPO + RLOO Advantage + Length-Penalty Reward  (arXiv:2502.04463)

```
Reward:  R(x,y) = 1{correct} · (1 − α · σ((len − μ_x) / σ_x))
RLOO:    A_i    = R_i − (1/(n−1)) · Σ_{j≠i} R_j
PPO:     L      = min(r_t·A_i, clip(r_t, 1±ε)·A_i) − β·KL(π||π_ref)
```


In [10]:
def extract_answer(text: str):
    """Extract final numeric answer."""
    patterns = [
        r'####\s*([\-\d\.\,]+)',
        r'\*\*Answer:\s*([\-\d\.\,]+)',
        r'Answer:\s*([\-\d\.\,]+)',
        r'=\s*([\-\d\.\,]+)\s*(?:<STOP>)?\s*$',
    ]
    for pat in patterns:
        m = re.search(pat, text, re.IGNORECASE | re.MULTILINE)
        if m:
            try:
                return str(float(m.group(1).replace(',', '')))
            except ValueError:
                pass
    nums = re.findall(r'[\-]?\d+\.?\d*', text)
    return str(float(nums[-1])) if nums else None


def is_correct(model_output: str, ground_truth: str) -> bool:
    pred = extract_answer(model_output)
    try:
        gt = str(float(ground_truth.replace(',', '')))
        return pred is not None and abs(float(pred) - float(gt)) < 1e-3
    except (ValueError, TypeError):
        return pred is not None and pred.strip() == ground_truth.strip()


print('Helper functions defined')


Helper functions defined


### Reward Function


In [11]:
class RewardModule:
    """
    R(x, y) = 1{y_correct} . (1 - alpha . sigmoid((len - mu_x) / sigma_x))
    Shortest correct answer -> highest reward. Wrong answers -> 0.
    """

    def __init__(self, alpha: float = 0.1):
        self.alpha = alpha

    @staticmethod
    def _sigmoid(x: float) -> float:
        return 1.0 / (1.0 + math.exp(-max(-500, min(500, x))))

    def compute_rewards(
        self,
        responses: List[str],
        ground_truth: str,
        alpha: Optional[float] = None,
    ) -> Tuple[List[float], List[bool]]:
        alpha       = alpha if alpha is not None else self.alpha
        correctness = [is_correct(r, ground_truth) for r in responses]
        lengths     = [len(r.split()) for r in responses]
        correct_lens = [l for l, c in zip(lengths, correctness) if c]
        if correct_lens:
            mu    = np.mean(correct_lens)
            sigma = max(np.std(correct_lens), 1.0)
        else:
            mu    = np.mean(lengths)
            sigma = max(np.std(lengths), 1.0)
        rewards = []
        for length, correct in zip(lengths, correctness):
            if not correct:
                rewards.append(0.0)
            else:
                z = (length - mu) / sigma
                f = self._sigmoid(z)
                rewards.append(float(np.clip(1.0 - alpha * f, 0.0, 1.0)))
        return rewards, correctness


rm_test = RewardModule(alpha=0.1)
test_r  = [
    'Step 1: 48/2=24. Total 48+24=72. #### 72 <STOP>',
    'Working step by step: April=48, May=48/2=24, total=72. #### 72 <STOP>',
    'The answer is 100 <STOP>',
]
rwds, corr = rm_test.compute_rewards(test_r, '72')
print('Reward Module sanity check:')
for r, c, resp in zip(rwds, corr, test_r):
    print(f'  Correct={c}, Reward={r:.3f} | {repr(resp[:65])}')


Reward Module sanity check:
  Correct=True, Reward=0.973 | 'Step 1: 48/2=24. Total 48+24=72. #### 72 <STOP>'
  Correct=True, Reward=0.927 | 'Working step by step: April=48, May=48/2=24, total=72. #### 72 <S'
  Correct=False, Reward=0.000 | 'The answer is 100 <STOP>'


### RLOO Advantage Estimator


In [12]:
class RLOOAdvantage:
    """
    A_i = R_i - (1/(n-1)) * sum(R_j for j != i)
    No separate critic -> 50-70% less GPU memory.
    """

    @staticmethod
    def compute(rewards: List[float]) -> List[float]:
        n          = len(rewards)
        assert n >= 2, 'Need at least 2 samples for RLOO'
        R          = np.array(rewards, dtype=np.float32)
        total      = R.sum()
        baselines  = (total - R) / (n - 1)
        advantages = R - baselines
        std = advantages.std()
        if std > 1e-8:
            advantages = advantages / std
        return advantages.tolist()


test_rewards = [0.9, 0.5, 0.0, 0.8, 0.6, 0.7, 0.0, 0.85]
advs = RLOOAdvantage.compute(test_rewards)
print('RLOO sanity check (n=8):')
print(f'  Rewards:    {[f"{r:.2f}" for r in test_rewards]}')
print(f'  Advantages: {[f"{a:.3f}" for a in advs]}')
print(f'  Best (0.90) -> A={advs[0]:.3f}  (positive = reinforce)')
print(f'  Wrong (0.00) -> A={advs[2]:.3f} (negative = suppress)')


RLOO sanity check (n=8):
  Rewards:    ['0.90', '0.50', '0.00', '0.80', '0.60', '0.70', '0.00', '0.85']
  Advantages: ['1.058', '-0.130', '-1.615', '0.761', '0.167', '0.464', '-1.615', '0.909']
  Best (0.90) -> A=1.058  (positive = reinforce)
  Wrong (0.00) -> A=-1.615 (negative = suppress)


### PPO Trainer


In [13]:
class PPOTrainerRLOO:
    def __init__(self, model, ref_model, tokenizer,
                 reward_module, alpha=0.1,
                 ppo_clip=0.2, kl_beta=0.02,
                 lr=1e-5, n_samples=8, max_new_tokens=256):
        self.model      = model
        self.ref_model  = ref_model
        self.tokenizer  = tokenizer
        self.reward     = reward_module
        self.alpha      = alpha
        self.eps        = ppo_clip
        self.beta       = kl_beta
        self.n          = n_samples
        self.max_new    = max_new_tokens
        self.optimizer  = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
        self.history    = defaultdict(list)

    @torch.no_grad()
    def _generate_one(self, input_ids):
        # Safety check: clamp input_ids to vocab size
        vocab_size = self.model.config.vocab_size
        input_ids = torch.clamp(input_ids, 0, vocab_size - 1)

        out = self.model.generate(
            input_ids,
            max_new_tokens=self.max_new,
            do_sample=True,
            temperature=0.8,
            top_p=0.9,
            pad_token_id=self.tokenizer.pad_token_id,
            return_dict_in_generate=True,
            output_scores=True,
            use_cache=False,
        )
        gen_ids = out.sequences[0, input_ids.shape[1]:]
        text    = self.tokenizer.decode(gen_ids, skip_special_tokens=True)
        scores  = torch.stack(out.scores, dim=0)
        lp_all  = F.log_softmax(scores, dim=-1)

        # Clamp gen_ids to prevent device-side assert during indexing
        gen_ids_clamped = torch.clamp(gen_ids, 0, lp_all.shape[-1] - 1)
        lp = lp_all[torch.arange(len(gen_ids)), gen_ids_clamped]
        return text, gen_ids, lp

    @torch.no_grad()
    def _ref_log_probs(self, full_ids, prompt_len):
        out     = self.ref_model(full_ids)
        gen_len = full_ids.shape[1] - prompt_len
        logits  = out.logits[0, prompt_len - 1: prompt_len - 1 + gen_len]
        lp      = F.log_softmax(logits, dim=-1)
        gen_ids = full_ids[0, prompt_len:]
        gen_ids_clamped = torch.clamp(gen_ids, 0, lp.shape[-1] - 1)
        return lp[torch.arange(len(gen_ids)), gen_ids_clamped]

    def _ppo_loss(self, full_ids, prompt_len, old_lp, advantage):
        gen_ids  = full_ids[0, prompt_len:]
        out      = self.model(full_ids)
        gen_len  = len(gen_ids)
        logits   = out.logits[0, prompt_len - 1: prompt_len - 1 + gen_len]
        new_lp_all = F.log_softmax(logits, dim=-1)
        gen_ids_clamped = torch.clamp(gen_ids, 0, new_lp_all.shape[-1] - 1)
        new_lp   = new_lp_all[torch.arange(gen_len), gen_ids_clamped]

        ratio    = torch.exp(new_lp - old_lp.to(DEVICE))
        A        = torch.tensor(advantage, dtype=torch.float32, device=DEVICE)
        surr     = torch.min(ratio * A, torch.clamp(ratio, 1 - self.eps, 1 + self.eps) * A)
        ref_lp   = self._ref_log_probs(full_ids, prompt_len)
        kl       = (new_lp - ref_lp).mean()
        loss     = -surr.mean() + self.beta * kl
        return loss, (-surr.mean()).item(), (self.beta * kl).item()

    def step(self, batch, step_idx, alpha=None):
        self.model.train()
        alpha = alpha if alpha is not None else self.alpha
        self.optimizer.zero_grad()
        total_loss = total_pl = total_kl = total_reward = 0.0
        total_correct = total_resp = 0
        all_lens = []

        for item in batch:
            question = item['question']
            gt       = item['final_answer']
            messages = [{'role': 'system', 'content': SYSTEM_PROMPT}, {'role': 'user', 'content': question}]
            prompt_text = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            enc = self.tokenizer(prompt_text, return_tensors='pt', truncation=True, max_length=MAX_SEQ_LENGTH // 2).to(DEVICE)
            prompt_len = enc['input_ids'].shape[1]

            texts, gen_ids_list, old_lps = [], [], []
            for _ in range(self.n):
                text, gen_ids, lp = self._generate_one(enc['input_ids'])
                texts.append(text)
                gen_ids_list.append(gen_ids)
                old_lps.append(lp)
                all_lens.append(len(text.split()))

            rewards, correctness = self.reward.compute_rewards(texts, gt, alpha=alpha)
            advantages = RLOOAdvantage.compute(rewards)

            n_total = len(batch) * self.n
            for gen_ids, old_lp, adv in zip(gen_ids_list, old_lps, advantages):
                full_ids = torch.cat([enc['input_ids'][0], gen_ids.to(DEVICE)]).unsqueeze(0)
                loss, pl, kl = self._ppo_loss(full_ids, prompt_len, old_lp, adv)
                (loss / n_total).backward()
                total_loss += loss.item()
                total_pl   += pl
                total_kl   += kl

            total_reward  += sum(rewards)
            total_correct += sum(correctness)
            total_resp    += len(texts)

        torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
        self.optimizer.step()
        metrics = {
            'loss':        total_loss  / max(total_resp, 1),
            'policy_loss': total_pl    / max(total_resp, 1),
            'kl':          total_kl    / max(total_resp, 1),
            'reward':      total_reward / max(total_resp, 1),
            'accuracy':    total_correct / max(total_resp, 1),
            'mean_tokens': np.mean(all_lens) if all_lens else 0.0,
        }
        for k, v in metrics.items():
            self.history[k].append(v)
        return metrics

### Extension 1 — Dynamic Alpha Curriculum (DAC)


In [14]:
class DynamicAlphaCurriculum:
    """
    alpha(x, k) = alpha_base + alpha_range . h(x) . s(k)
    h(x) = 1 - pass@1 rate (online hardness estimate)
    s(k) = cosine curriculum schedule in [0,1]
    Easy problems -> high alpha -> strong compression
    Hard problems -> low  alpha -> protect accuracy
    """

    def __init__(self, alpha_base=0.1, alpha_range=0.15, total_steps=100):
        self.alpha_base  = alpha_base
        self.alpha_range = alpha_range
        self.total_steps = total_steps
        self._attempts   = {}
        self._correct    = {}

    def update(self, question: str, correct: bool):
        k = question[:60]
        self._attempts[k] = self._attempts.get(k, 0) + 1
        if correct:
            self._correct[k] = self._correct.get(k, 0) + 1

    def hardness(self, question: str) -> float:
        k     = question[:60]
        n     = self._attempts.get(k, 0)
        c     = self._correct.get(k, 0)
        pass1 = c / n if n > 0 else 0.5
        return 1.0 - pass1

    def schedule(self, step: int) -> float:
        return 0.5 * (1 - math.cos(math.pi * step / max(self.total_steps, 1)))

    def get_alpha(self, question: str, step: int) -> float:
        return float(np.clip(
            self.alpha_base + self.alpha_range * self.hardness(question) * self.schedule(step),
            0.0, 0.5
        ))


print('DynamicAlphaCurriculum (DAC) defined')


DynamicAlphaCurriculum (DAC) defined


### Extension 2 — Multi-Objective Reward Shaping (MORS)


In [15]:
from sentence_transformers import SentenceTransformer, util as st_util

class MORSReward:
    """
    R_total = 0.70.R_corr + 0.15.R_len + 0.10.R_qual + 0.05.R_fmt
    """
    W = {'corr': 0.70, 'len': 0.15, 'qual': 0.10, 'fmt': 0.05}

    def __init__(self):
        self._embedder = None

    @property
    def embedder(self):
        if self._embedder is None:
            self._embedder = SentenceTransformer('all-MiniLM-L6-v2')
        return self._embedder

    def r_corr(self, output, gt):       return 1.0 if is_correct(output, gt) else 0.0

    def r_len(self, output, outputs, gt):
        correct = [o for o in outputs if is_correct(o, gt)]
        if not correct or not is_correct(output, gt): return 0.0
        mn = min(len(o.split()) for o in correct)
        mx = max(len(o.split()) for o in correct)
        if mx == mn: return 1.0
        return 1.0 - (len(output.split()) - mn) / (mx - mn + 1e-8)

    def r_qual(self, output, reference):
        e_out = self.embedder.encode(output,    convert_to_tensor=True)
        e_ref = self.embedder.encode(reference, convert_to_tensor=True)
        return max(0.0, float(st_util.cos_sim(e_out, e_ref).item()))

    def r_fmt(self, output):
        lower = output.lower()
        s  = 0.5 if any(k in lower for k in ['step', 'first', 'then', 'finally']) else 0.0
        s += 0.5 if any(k in lower for k in ['####', '<stop>', 'answer:']) else 0.0
        return s

    def compute(self, output, outputs, gt, reference=''):
        rc = self.r_corr(output, gt)
        rl = self.r_len(output, outputs, gt)
        rq = self.r_qual(output, reference or gt)
        rf = self.r_fmt(output)
        total = self.W['corr']*rc + self.W['len']*rl + self.W['qual']*rq + self.W['fmt']*rf
        return total, {'R_corr': rc, 'R_len': rl, 'R_qual': rq, 'R_fmt': rf, 'R_total': total}


print('MORSReward defined')


MORSReward defined


In [16]:
rl_prompts = []
for conv in conversations:
    msgs     = conv.get('messages', [])
    user_msg = next((m['content'] for m in msgs if m['role'] == 'user'),  None)
    asst_msg = next((m['content'] for m in msgs if m['role'] == 'assistant'), None)
    if user_msg and asst_msg:
        m = re.match(r'.*####\s*([\d\.\,\-]+)', asst_msg, re.DOTALL)
        final_ans = m.group(1).strip().replace(',', '') if m else asst_msg.strip()
        rl_prompts.append({
            'question':     user_msg.strip(),
            'answer':       asst_msg.strip(),
            'final_answer': final_ans,
        })

print(f'RL prompts: {len(rl_prompts)}')
print(f'  Sample Q: {rl_prompts[0]["question"][:70]}...')
print(f'  Answer:   {rl_prompts[0]["final_answer"]}')


RL prompts: 300
  Sample Q: Natalia sold clips to 48 of her friends in April, and then she sold ha...
  Answer:   72


In [17]:
from transformers import AutoTokenizer
from unsloth import FastLanguageModel
import random, numpy as np
import torch

# Updated config for T4 stability
RL_CONFIG = {
    'n_steps':        10,
    'batch_size':     2,      # Reduced for memory
    'n_samples':      4,      # Reduced for memory
    'alpha':          0.1,
    'ppo_clip':       0.2,
    'kl_beta':        0.02,
    'lr':             1e-5,
    'max_new_tokens': 128,
    'use_dac':        True,
    'log_every':      10,
}

print('Cleaning GPU memory...')
torch.cuda.empty_cache()

print(f'Loading models from {SFT_OUTPUT}...')
rl_tokenizer = AutoTokenizer.from_pretrained(SFT_OUTPUT, trust_remote_code=True)

# Load Policy
rl_policy, _ = FastLanguageModel.from_pretrained(
    model_name=SFT_OUTPUT,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    resize_model_vocab=151666
)

rl_policy = FastLanguageModel.get_peft_model(
    rl_policy,
    r=16, lora_alpha=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    use_gradient_checkpointing='unsloth',
    random_state=42,
)

# Load Reference
rl_ref, _ = FastLanguageModel.from_pretrained(
    model_name=SFT_OUTPUT,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    resize_model_vocab=151666
)
rl_ref.eval()

ppo_trainer = PPOTrainerRLOO(
    model=rl_policy,
    ref_model=rl_ref,
    tokenizer=rl_tokenizer,
    reward_module=RewardModule(alpha=RL_CONFIG['alpha']),
    alpha=RL_CONFIG['alpha'],
    ppo_clip=RL_CONFIG['ppo_clip'],
    kl_beta=RL_CONFIG['kl_beta'],
    lr=RL_CONFIG['lr'],
    n_samples=RL_CONFIG['n_samples'],
    max_new_tokens=RL_CONFIG['max_new_tokens'],
)

print('RL training components re-initialized with safety fixes.')

Cleaning GPU memory...
Loading models from ./fine_tuned_qwen25_3b...
==((====))==  Unsloth 2026.3.18: Fast Qwen2 patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth: Already have LoRA adapters! We shall skip this step.


==((====))==  Unsloth 2026.3.18: Fast Qwen2 patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
RL training components re-initialized with safety fixes.


In [18]:
from transformers import AutoTokenizer
from unsloth import FastLanguageModel

print('Loading SFT checkpoint for RL fine-tuning...')

# ✅ 1. Load EXACT tokenizer from SFT — preserve the custom pad token
rl_tokenizer = AutoTokenizer.from_pretrained(
    SFT_OUTPUT,
    trust_remote_code=True
)

# ✅ 2. DO NOT override pad_token — it was saved correctly in SFT.
#    Just confirm it's set (it should already be '<|pad_token|>').
assert rl_tokenizer.pad_token is not None, \
    "pad_token missing from saved tokenizer — check SFT save step"

VOCAB_SIZE = len(rl_tokenizer)   # e.g., 151644 + 1 = 151645
print(f"Tokenizer vocab size: {VOCAB_SIZE}")

# ✅ 3. Load policy model
rl_policy, _ = FastLanguageModel.from_pretrained(
    model_name=SFT_OUTPUT,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

# ✅ 4. Resize to EXACT tokenizer size — must match SFT checkpoint
rl_policy.resize_token_embeddings(VOCAB_SIZE)

# ✅ 5. Sync config token IDs — use pad_token_id, NOT eos_token_id
rl_policy.config.pad_token_id = rl_tokenizer.pad_token_id
rl_policy.config.eos_token_id = rl_tokenizer.eos_token_id
rl_policy.config.bos_token_id = rl_tokenizer.bos_token_id
rl_policy.config.use_cache = False   # 🔴 IMPORTANT for training

# ✅ 6. Apply LoRA
rl_policy = FastLanguageModel.get_peft_model(
    rl_policy,
    r=16, lora_alpha=16, lora_dropout=0,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
FastLanguageModel.for_training(rl_policy)

print('Loading frozen reference model...')

# ✅ 7. Load reference model identically
rl_ref, _ = FastLanguageModel.from_pretrained(
    model_name=SFT_OUTPUT,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)
rl_ref.resize_token_embeddings(VOCAB_SIZE)   # ← Same VOCAB_SIZE
rl_ref.eval()
for p in rl_ref.parameters():
    p.requires_grad_(False)

rl_ref.config.pad_token_id = rl_tokenizer.pad_token_id
rl_ref.config.eos_token_id = rl_tokenizer.eos_token_id
rl_ref.config.use_cache = False

print(f"✅ RL setup complete. Vocab size: {VOCAB_SIZE}")

Loading SFT checkpoint for RL fine-tuning...
Tokenizer vocab size: 151666
==((====))==  Unsloth 2026.3.18: Fast Qwen2 patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


RuntimeError: Error(s) in loading state_dict for PeftModelForCausalLM:
	size mismatch for base_model.model.model.embed_tokens.weight: copying a param with shape torch.Size([151666, 2048]) from checkpoint, the shape in current model is torch.Size([151936, 2048]).
	size mismatch for base_model.model.lm_head.weight: copying a param with shape torch.Size([151666, 2048]) from checkpoint, the shape in current model is torch.Size([151936, 2048]).

In [ ]:
from transformers import AutoTokenizer
from unsloth import FastLanguageModel
import random, numpy as np

RL_CONFIG = {
    'n_steps':        100,
    'batch_size':     4,
    'n_samples':      8,
    'alpha':          0.1,
    'ppo_clip':       0.2,
    'kl_beta':        0.02,
    'lr':             1e-5,
    'max_new_tokens': 256,
    'use_dac':        True,
    'log_every':      10,
}

print('Loading SFT checkpoint for RL fine-tuning...')

rl_tokenizer = AutoTokenizer.from_pretrained(SFT_OUTPUT, trust_remote_code=True)

# We specify resize_model_vocab=151666 to match the SFT checkpoint precisely
rl_policy, _ = FastLanguageModel.from_pretrained(
    model_name=SFT_OUTPUT,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
    resize_model_vocab=151666
)

# Apply LoRA
rl_policy = FastLanguageModel.get_peft_model(
    rl_policy,
    r=16, lora_alpha=16, lora_dropout=0,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    bias='none', outcome_map=None,
    use_gradient_checkpointing='unsloth', # Corrected parameter
    random_state=42,
)

FastLanguageModel.for_training(rl_policy)
rl_policy.config.use_cache = False

print('Loading frozen reference model...')
rl_ref, _ = FastLanguageModel.from_pretrained(
    model_name=SFT_OUTPUT,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
    resize_model_vocab=151666
)
rl_ref.eval()

ppo_trainer = PPOTrainerRLOO(
    model=rl_policy,
    ref_model=rl_ref,
    tokenizer=rl_tokenizer,
    reward_module=RewardModule(alpha=RL_CONFIG['alpha']),
    alpha=RL_CONFIG['alpha'],
    ppo_clip=RL_CONFIG['ppo_clip'],
    kl_beta=RL_CONFIG['kl_beta'],
    lr=RL_CONFIG['lr'],
    n_samples=RL_CONFIG['n_samples'],
    max_new_tokens=RL_CONFIG['max_new_tokens'],
)

print('RL Components ready. Run this cell to start training.')

In [ ]:
import time
torch.cuda.empty_cache()

dac = DynamicAlphaCurriculum(total_steps=RL_CONFIG['n_steps'])

print(f"Restarting RL Training for {RL_CONFIG['n_steps']} steps...")
t0_train = time.time()

try:
    for step in range(1, RL_CONFIG['n_steps'] + 1):
        batch = random.sample(rl_prompts, RL_CONFIG['batch_size'])
        current_alpha = dac.get_alpha(batch[0]['question'], step) if RL_CONFIG['use_dac'] else RL_CONFIG['alpha']

        metrics = ppo_trainer.step(batch, step, alpha=current_alpha)
        dac.update(batch[0]['question'], metrics['accuracy'] > 0.5)

        if step % RL_CONFIG['log_every'] == 0:
            print(f"Step {step:3d} | Loss: {metrics['loss']:.4f} | Reward: {metrics['reward']:.3f} | Acc: {metrics['accuracy']:.2f} | Alpha: {current_alpha:.3f}")
except Exception as e:
    print(f"Training stopped at step {step} due to error: {e}")

t1_train = time.time()
print(f'Training finished in {(t1_train - t0_train)/60:.2f} minutes.')

In [ ]:
print('Running Final Evaluation Suite...')
# Ensure the model is in inference mode and memory is cleared
FastLanguageModel.for_inference(rl_policy)
torch.cuda.empty_cache()

eval_suite = EvaluationSuite()
eval_results = eval_suite.evaluate_all(
    model=rl_policy,
    tokenizer=rl_tokenizer,
    test_prompts=rl_prompts,
    n=10, # Minimal set for T4 stability
    k=2   # Minimal k to prevent OOM
)
display(eval_results)

---
## Stage 3 · Automatic Evaluation — All 5 Metrics

| Metric | Target |
|--------|--------|
| Exact Match Accuracy | >= 78% |
| Token Reduction | 40–50% vs baseline |
| Faithfulness Score | >= 0.82 |
| Response Consistency (k=5) | >= 0.85 |
| Latency Reduction | ~2–3× |


In [ ]:
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

class EvaluationSuite:
    """Automatic evaluation suite for the efficient reasoning model."""

    def __init__(self):
        print('Loading sentence-transformer for faithfulness metric...')
        self.embedder = SentenceTransformer('all-MiniLM-L6-v2')

    def exact_match_accuracy(self, outputs, ground_truths):
        correct = sum(is_correct(o, g) for o, g in zip(outputs, ground_truths))
        acc     = correct / len(outputs)
        return {'metric': 'Exact Match Accuracy', 'value': acc, 'target': 0.78,
                'pass': acc >= 0.78, 'correct': correct, 'total': len(outputs)}

    def token_reduction(self, efficient, baseline):
        mean_eff  = np.mean([len(o.split()) for o in efficient])
        mean_base = np.mean([len(o.split()) for o in baseline])
        red = 1.0 - mean_eff / (mean_base + 1e-8)
        return {'metric': 'Token Reduction', 'value': red, 'target': 0.40,
                'pass': red >= 0.40, 'reduction_pct': red * 100,
                'mean_efficient_tokens': mean_eff, 'mean_baseline_tokens': mean_base}

    def faithfulness_score(self, outputs, references):
        scores = []
        for out, ref in zip(outputs, references):
            e_out = self.embedder.encode(out, convert_to_tensor=True)
            e_ref = self.embedder.encode(ref, convert_to_tensor=True)
            scores.append(max(0.0, float(st_util.cos_sim(e_out, e_ref).item())))
        mean_f = np.mean(scores)
        return {'metric': 'Faithfulness Score', 'value': mean_f, 'target': 0.82,
                'pass': mean_f >= 0.82}

    def response_consistency(self, model, tokenizer, questions, k=5):
        rates = []
        for q in questions:
            messages = [
                {'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user',   'content': q},
            ]
            text   = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=512).to(DEVICE)
            answers = []
            for _ in range(k):
                with torch.no_grad():
                    out = model.generate(
                        **inputs, max_new_tokens=200,
                        do_sample=True, temperature=0.7, use_cache=True,
                        pad_token_id=tokenizer.pad_token_id,
                    )
                gen = tokenizer.decode(out[0, inputs['input_ids'].shape[1]:], skip_special_tokens=True)
                ans = extract_answer(gen)
                if ans:
                    answers.append(ans)
            if len(answers) >= 2:
                pairs = [(a, b) for i, a in enumerate(answers)
                         for j, b in enumerate(answers) if i < j]
                try:
                    agree = sum(abs(float(a) - float(b)) < 1e-3 for a, b in pairs)
                    rates.append(agree / len(pairs))
                except (ValueError, TypeError):
                    rates.append(0.0)
            else:
                rates.append(0.0)
        mean_c = np.mean(rates) if rates else 0.0
        return {'metric': 'Response Consistency', 'value': mean_c, 'target': 0.85,
                'pass': mean_c >= 0.85, 'k': k}

    def latency_reduction(self, efficient, baseline):
        ratio = np.mean([len(o.split()) for o in baseline]) / (
            np.mean([len(o.split()) for o in efficient]) + 1e-8
        )
        return {'metric': 'Latency Reduction', 'value': ratio, 'target': 2.0,
                'pass': ratio >= 2.0, 'speedup_factor': ratio}

    def evaluate_all(self, model, tokenizer, test_prompts, n=50, k=5):
        FastLanguageModel.for_inference(model)
        test_prompts = test_prompts[:n]
        print(f'\nGenerating {n} efficient + baseline outputs...')
        efficient_outputs, baseline_outputs = [], []
        for item in test_prompts:
            messages = [
                {'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user',   'content': item['question']},
            ]
            text   = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=512).to(DEVICE)
            pl = inputs['input_ids'].shape[1]
            with torch.no_grad():
                out_eff  = model.generate(**inputs, max_new_tokens=256,
                    do_sample=False, use_cache=True, pad_token_id=tokenizer.pad_token_id)
                out_base = model.generate(**inputs, max_new_tokens=512,
                    do_sample=True, temperature=1.0, use_cache=True, pad_token_id=tokenizer.pad_token_id)
            efficient_outputs.append(tokenizer.decode(out_eff[0, pl:],  skip_special_tokens=True))
            baseline_outputs.append( tokenizer.decode(out_base[0, pl:], skip_special_tokens=True))

        gt_list  = [item['final_answer'] for item in test_prompts]
        ref_list = [item['answer']       for item in test_prompts]

        print('Computing metrics...')
        results = {}
        results['accuracy']        = self.exact_match_accuracy(efficient_outputs, gt_list)
        results['token_reduction'] = self.token_reduction(efficient_outputs, baseline_outputs)
        results['faithfulness']    = self.faithfulness_score(efficient_outputs, ref_list)
        results['consistency']     = self.response_consistency(
            model, tokenizer, [item['question'] for item in test_prompts[:20]], k=k
        )
        results['latency']         = self.latency_reduction(efficient_outputs, baseline_outputs)

        print('\n' + '=' * 62)
        print(f'{"METRIC":<28} {"ACHIEVED":>10} {"TARGET":>10} {"PASS":>8}')
        print('-' * 62)
        FMT = {
            'accuracy':        lambda v, t: (f'{v*100:.1f}%', f'>={t*100:.0f}%'),
            'token_reduction': lambda v, t: (f'{v*100:.1f}%', f'>={t*100:.0f}%'),
            'faithfulness':    lambda v, t: (f'{v:.4f}',      f'>={t:.2f}'),
            'consistency':     lambda v, t: (f'{v:.4f}',      f'>={t:.2f}'),
            'latency':         lambda v, t: (f'{v:.2f}x',     f'>={t:.0f}x'),
        }
        for key, res in results.items():
            vs, ts = FMT[key](res['value'], res['target'])
            flag   = 'PASS' if res['pass'] else 'FAIL'
            print(f"{res['metric']:<28} {vs:>10} {ts:>10} {flag:>8}")
        print('=' * 62)
        tr = results['token_reduction']
        print(f'\nToken summary:')
        print(f'  Efficient: {tr["mean_efficient_tokens"]:.0f} tokens/response')
        print(f'  Baseline:  {tr["mean_baseline_tokens"]:.0f} tokens/response')
        print(f'  Reduction: {tr["reduction_pct"]:.1f}%')
        print(f'  Speedup:   {results["latency"]["speedup_factor"]:.2f}x')
        return results


print('EvaluationSuite defined')


In [ ]:
eval_suite   = EvaluationSuite()
eval_results = eval_suite.evaluate_all(
    model=rl_policy,
    tokenizer=rl_tokenizer,
    test_prompts=rl_prompts,
    n=50,
    k=5,
)


---
## Stage 4 · Visualisations


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_training_curves(trainer):
    # Ensure history values are standard floats, not CUDA tensors
    h = {k: [float(x) if torch.is_tensor(x) else x for x in v] for k, v in trainer.history.items()}
    keys   = ['loss', 'reward', 'accuracy', 'kl', 'mean_tokens', 'policy_loss']
    labels = ['Total Loss', 'Mean Reward', 'Accuracy', 'KL Divergence', 'Mean Tokens', 'Policy Loss']
    colors = ['#E74C3C', '#2ECC71', '#3498DB', '#9B59B6', '#F39C12', '#1ABC9C']

    fig, axes = plt.subplots(2, 3, figsize=(16, 8))
    fig.suptitle('RL Training — Qwen2.5-3B · PPO + RLOO', fontsize=14, fontweight='bold')

    for ax, key, label, color in zip(axes.flatten(), keys, labels, colors):
        vals = h.get(key, [])
        if not vals:
            ax.set_visible(False)
            continue
        ax.plot(vals, color=color, linewidth=1.5, alpha=0.4)
        if len(vals) > 5:
            w = max(1, len(vals) // 5)
            sm = np.convolve(vals, np.ones(w) / w, mode='valid')
            ax.plot(range(len(sm)), sm, color=color, linewidth=2.5)
        ax.set_title(label, fontweight='bold')
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

def plot_eval_results(eval_results):
    if not eval_results: return
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    names, achieved, targets = [], [], []
    for key, res in eval_results.items():
        names.append(res['metric'])
        v, t = float(res['value']), float(res['target'])
        if key in ('accuracy', 'token_reduction'): v, t = v * 100, t * 100
        achieved.append(v)
        targets.append(t)

    x = np.arange(len(names))
    ax1.bar(x - 0.2, achieved, 0.4, label='Achieved', color='#2ECC71')
    ax1.bar(x + 0.2, targets, 0.4, label='Target', color='#3498DB')
    ax1.set_xticks(x)
    ax1.set_xticklabels(names, rotation=45)
    ax1.legend()
    plt.tight_layout()
    plt.show()

# Safely call plots
plot_training_curves(ppo_trainer)
if 'eval_results' in locals():
    plot_eval_results(eval_results)

---
## Stage 5 · Inference Demo


In [ ]:
import time
FastLanguageModel.for_inference(rl_policy)

DEMO_QUESTIONS = [
    'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?',
    'Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. How much did she earn?',
    'Betty is saving money for a new wallet which costs $100. Betty has only half of the money she needs. Her parents decided to give her $15 for that purpose, and her grandparents twice as much as her parents. How much more money does Betty need to buy the wallet?',
]

print('=' * 60)
print('EFFICIENT REASONING DEMO — Qwen2.5-3B (RL-trained)')
print('=' * 60)
for q in DEMO_QUESTIONS:
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': q},
    ]
    text   = rl_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = rl_tokenizer(text, return_tensors='pt').to(DEVICE)
    t0 = time.time()
    with torch.no_grad():
        out = rl_policy.generate(
            **inputs, max_new_tokens=256, do_sample=False, use_cache=True,
            pad_token_id=rl_tokenizer.pad_token_id,
        )
    elapsed  = time.time() - t0
    response = rl_tokenizer.decode(out[0, inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    tokens   = len(response.split())
    answer   = extract_answer(response)
    print(f'\nQ: {q}')
    print(f'A: {response}')
    print(f'   Tokens: {tokens} | Latency: {elapsed:.2f}s | Answer: {answer}')
    print('-' * 60)


---
## Stage 6 · Summary Report


In [ ]:
import json as _json

summary = {}
for k, v in eval_results.items():
    summary[k] = {
        ky: float(va) if isinstance(va, (float, np.floating)) else va
        for ky, va in v.items() if not isinstance(va, list)
    }
summary['model']     = MODEL_ID
summary['rl_config'] = RL_CONFIG
summary['timestamp'] = time.strftime('%Y-%m-%d %H:%M:%S')

with open('summary_report_qwen.json', 'w') as f:
    _json.dump(summary, f, indent=2)

print('Summary report saved -> summary_report_qwen.json')
print(_json.dumps({k: v for k, v in summary.items() if k not in ('rl_config',)}, indent=2))


---
## Appendix · Paper vs Implementation Comparison

| Component | Paper (arXiv:2502.04463) | This Implementation |
|-----------|--------------------------|---------------------|
| Base model | Qwen-7B / DeepSeek-R1 | **Qwen2.5-3B-Instruct** |
| SFT | Full fine-tuning | **Unsloth LoRA (r=16, 4-bit)** |
| Reward | Length penalty | **Same (prompt-normalised sigmoid)** |
| Advantage | RLOO | **Same (Leave-One-Out)** |
| PPO clip | ε=0.2 | **Same** |
| KL penalty | β coefficient | **Same** |
| Extension | — | **DAC + MORS** |
| Dataset | MATH/GSM8K | **300 GSM8K (TuneKit)** |

**Why Qwen2.5-3B over the original Phi-4-mini / Mistral-7B?**
- Smaller but stronger math reasoning (82.7% GSM8K zero-shot vs Mistral-7B's ~60%)
- Fits T4 with more headroom for RL training (policy + frozen ref + optimizer states)
- Aligned chat template — consistent with SFT → RL pipeline
- Excellent token efficiency baseline makes length-penalty reward more discriminative
